# Smart Traffic Management: density evaluation and demo on real traffic video

Two experiments with the real system (YOLOv8 + tracker + per-lane analysis + signal controller):

1. **Is the density classification right?** The [UCSD highway traffic videos](https://www.kaggle.com/datasets/aryashah2k/highway-traffic-videos-dataset) (CC0) have a human label for each clip: *light*, *medium* or *heavy* traffic. The system counts vehicles in each clip; thresholds are fitted on the training clips and tested on unseen test clips.
2. **Demo on a hard scene:** a rain / snow intersection video from the [AAU RainSnow dataset](https://www.kaggle.com/datasets/aalborguniversity/aau-rainsnow) (CC BY 4.0, Aalborg University), saved as annotated frames, a GIF and a summary.

**Kaggle settings:** Accelerator → **GPU T4 x2**, Internet → **On**, and add both datasets as input. Run time: about 20 minutes.

In [ ]:
!pip install -q ultralytics
!git clone -q https://github.com/pavanadityav-ai/smart-traffic-management.git /kaggle/working/app
import glob, json, os, re, sys
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
sys.path.insert(0, "/kaggle/working/app")
from traffic.detection import YoloDetector
from traffic.pipeline import TrafficPipeline
from traffic.storage import Storage

OUT = Path("/kaggle/working/results")
OUT.mkdir(exist_ok=True)
detector = YoloDetector("yolov8s.pt", confidence=0.3)

## 1. Density classification against human labels (UCSD highway clips)
### Find the clips and their labels

In [ ]:
readme = sorted(glob.glob("/kaggle/input/**/README_TRAFFICDB", recursive=True))[0]
UCSD = Path(readme).parent
print(open(readme, errors="ignore").read()[:1500])
for name in ("info.txt", "ImageMaster", "EvalSet_train", "EvalSet_test"):
    p = UCSD / name
    if p.exists():
        print(f"----- {name} -----")
        print(open(p, errors="ignore").read()[:600])
videos = sorted(glob.glob(f"{UCSD}/**/*.avi", recursive=True))
print(len(videos), "videos, for example", videos[:2])

In [ ]:
# Labels: find lines that mention a clip name and one of light / medium / heavy.
stems = {Path(v).stem: v for v in videos}
labels = {}
for name in ("ImageMaster", "info.txt"):
    p = UCSD / name
    if not p.exists():
        continue
    for line in open(p, errors="ignore"):
        cls = re.search(r"\b(light|medium|heavy)\b", line, re.I)
        clip = next((s for s in re.findall(r"[\w\-]+", line) if s in stems), None)
        if cls and clip:
            labels.setdefault(clip, cls.group(1).lower())
if len(labels) < len(videos) // 2:  # fall back to the MATLAB file
    from scipy.io import loadmat
    mat = loadmat(str(UCSD / "ImageMaster.mat"), squeeze_me=True, struct_as_record=False)
    key = [k for k in mat if not k.startswith("__")][0]
    for entry in np.atleast_1d(mat[key]):
        fields = {f: getattr(entry, f) for f in entry._fieldnames}
        text = " ".join(str(v) for v in fields.values())
        cls = re.search(r"\b(light|medium|heavy)\b", text, re.I)
        clip = next((s for s in re.findall(r"[\w\-]+", text) if s in stems), None)
        if cls and clip:
            labels[clip] = cls.group(1).lower()
print(len(labels), "labelled clips:", Counter(labels.values()))
assert len(labels) > 100, "label parsing failed: look at the files printed above"

### Count vehicles in every clip
For each clip the pipeline runs frame by frame; the feature is the **average number of tracked vehicles on screen**.

In [ ]:
def clip_feature(path):
    pipe = TrafficPipeline(detector, n_lanes=1)
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 10
    counts, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        i += 1
        _, state = pipe.process(frame, t=i / fps)
        counts.append(state["analysis"]["total"])
    cap.release()
    return float(np.mean(counts[3:])) if len(counts) > 3 else float(np.mean(counts or [0]))

first = cv2.VideoCapture(videos[0]); ok, f0 = first.read(); first.release()
print("frame size:", f0.shape)
features = {}
for k, clip in enumerate(sorted(labels)):
    features[clip] = clip_feature(stems[clip])
    if k % 50 == 0:
        print(k, clip, labels[clip], round(features[clip], 2))

### Fit the two thresholds on training clips, test on unseen clips

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

ORDER = ["light", "medium", "heavy"]
clips = sorted(labels)
rng = np.random.default_rng(0)
test_set = set()
for cls in ORDER:  # stratified 50 / 50 split, fixed seed
    members = [c for c in clips if labels[c] == cls]
    test_set.update(rng.choice(members, size=len(members) // 2, replace=False))
train = [c for c in clips if c not in test_set]
test = [c for c in clips if c in test_set]

def predict(x, t1, t2):
    return "light" if x < t1 else ("medium" if x < t2 else "heavy")

values = sorted({round(features[c], 3) for c in train})
best = (0, None, None)
for i, t1 in enumerate(values):
    for t2 in values[i:]:
        acc = np.mean([predict(features[c], t1, t2) == labels[c] for c in train])
        if acc > best[0]:
            best = (acc, t1, t2)
train_acc, T1, T2 = best
y_true = [labels[c] for c in test]
y_pred = [predict(features[c], T1, T2) for c in test]
majority = Counter(labels[c] for c in train).most_common(1)[0][0]
density_eval = {
    "dataset": "UCSD highway traffic videos (CC0), labels light / medium / heavy",
    "clips": len(clips), "train_clips": len(train), "test_clips": len(test),
    "feature": "average number of tracked vehicles per frame (YOLOv8s + IoU tracker)",
    "thresholds": {"medium_from": T1, "heavy_from": T2},
    "train_accuracy": round(float(train_acc), 4),
    "test_accuracy": round(accuracy_score(y_true, y_pred), 4),
    "test_macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
    "majority_class_baseline": round(float(np.mean([y == majority for y in y_true])), 4),
    "confusion_matrix": {"labels": ORDER, "rows_true_cols_pred": confusion_matrix(y_true, y_pred, labels=ORDER).tolist()},
    "mean_feature_by_class": {c: round(float(np.mean([features[k] for k in clips if labels[k] == c])), 2) for c in ORDER},
}
print(json.dumps(density_eval, indent=2))
json.dump(density_eval, open(OUT / "density_eval.json", "w"), indent=2)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
for y, cls in enumerate(ORDER):
    xs = [features[c] for c in clips if labels[c] == cls]
    ax.scatter(xs, np.full(len(xs), y) + rng.uniform(-0.15, 0.15, len(xs)), s=12, label=cls)
for t in (T1, T2):
    ax.axvline(t, color="k", linestyle="--", linewidth=1)
ax.set_yticks(range(3), ORDER)
ax.set_xlabel("average vehicles on screen")
ax.set_title("Vehicle count per clip vs human traffic label (dashed: fitted thresholds)")
plt.tight_layout()
plt.savefig(OUT / "density_eval.png", dpi=120)
plt.show()

## 2. Demo: rain and snow at a real intersection (AAU RainSnow)

In [ ]:
video = sorted(glob.glob("/kaggle/input/**/Egensevej-1/cam1.mkv", recursive=True))[0]
cap = cv2.VideoCapture(video)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
print(video, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), "frames at", fps, "fps",
      int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
ok, raw = cap.read()
cv2.imwrite(str(OUT / "demo_raw_first_frame.jpg"), raw)
cap.release()

In [ ]:
import time
from PIL import Image

LANES = 3
SECONDS = 40
db_path = OUT / "demo.db"
if db_path.exists():
    db_path.unlink()
db = Storage(str(db_path))
run = db.start_run("AAU RainSnow Egensevej-1 cam1", "yolov8s", LANES)
pipe = TrafficPipeline(detector, n_lanes=LANES, storage=db, run_id=run)

cap = cv2.VideoCapture(video)
h, w = raw.shape[:2]
writer = cv2.VideoWriter(str(OUT / "demo_annotated.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
gif_frames, flag_counts, times, i = [], Counter(), [], 0
while i < SECONDS * fps:
    ok, frame = cap.read()
    if not ok:
        break
    i += 1
    t0 = time.perf_counter()
    annotated, state = pipe.process(frame, t=i / fps)
    times.append(time.perf_counter() - t0)
    writer.write(annotated)
    flag_counts.update(k for k, v in state["flags"].items() if v is True)
    if i % 3 == 0 and i <= 12 * fps:
        small = cv2.resize(annotated, (480, int(480 * h / w)))
        gif_frames.append(Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)))
    if i in (int(5 * fps), int(20 * fps), int(35 * fps)):
        cv2.imwrite(str(OUT / f"demo_frame_{int(i / fps)}s.jpg"), annotated, [cv2.IMWRITE_JPEG_QUALITY, 85])
cap.release()
writer.release()
gif_frames[0].save(OUT / "demo.gif", save_all=True, append_images=gif_frames[1:], duration=int(3000 / fps), loop=0)
db.finish_run(run, "finished", pipe.frames, len(times) / sum(times), pipe.total_ms / pipe.frames, dict(pipe.tracker.unique_counts))

summary = {
    "video": "AAU RainSnow, Egensevej-1 cam1 (CC BY 4.0)", "seconds_analysed": round(i / fps, 1), "frames": i,
    "model": "YOLOv8s (COCO)", "device": __import__("torch").cuda.get_device_name(0),
    "avg_inference_ms": round(pipe.total_ms / pipe.frames, 1),
    "avg_pipeline_fps": round(len(times) / sum(times), 1), "video_fps": fps,
    "unique_vehicles": dict(pipe.tracker.unique_counts),
    "density_seconds": db.summary(run)["density_seconds"],
    "lanes": db.summary(run)["lanes"],
    "frames_flagged_pct": {k: round(100 * v / i, 1) for k, v in flag_counts.items()},
}
print(json.dumps(summary, indent=2))
json.dump(summary, open(OUT / "demo_summary.json", "w"), indent=2)

In [ ]:
from IPython.display import Image as Show, display
for p in sorted(OUT.glob("demo_frame_*.jpg")):
    display(Show(str(p), width=800))

## Observations
*(Write 3–4 points after the run: test accuracy vs the majority baseline, which classes get confused, how the system behaves in rain / snow, and whether it runs in real time on the T4.)*